In [1]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.get_device_name(0))  #

True
NVIDIA GeForce RTX 3080 Ti


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Access variables
token = os.getenv("hf_token")

In [3]:
from transformers import AutoTokenizer, LlamaForCausalLM, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, # 8 bit uses too much memory
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
model_id_base = "meta-llama/Meta-Llama-3.1-8B" # General model
model_id_instruct = "meta-llama/Meta-Llama-3.1-8B-Instruct" # Instruct model


# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id_base)

# Load the model with quantization and auto device mapping
model = AutoModelForCausalLM.from_pretrained(
    model_id_base,
    quantization_config=bnb_config,
    device_map="auto"  # Automatically uses GPU, with CPU fallback if necessary
)

model.gradient_checkpointing_enable()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
from datasets import load_dataset

# Load dataset in streaming mode to process line by line (saves memory)
dataset = load_dataset(
        "json",
        data_files={"train": "data/krusty_krab.jsonl"},
        split="train",
        streaming=True  # Enables streaming mode
    )
dataset

IterableDataset({
    features: ['index', 'text', 'date_utc', 'is_from_me', 'cache_has_attachments', 'message_id', 'reaction', 'mime_type', 'name', 'members'],
    n_shards: 1
})

In [5]:
import json
from tqdm import tqdm
import os

tokenized_output_file = "data/tokenized_output.json"
data_output_file = "data/output.json"

if os.path.exists(tokenized_output_file):
    os.remove(tokenized_output_file)
          
if os.path.exists(data_output_file):
    os.remove(data_output_file)        

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use EOS token for padding

# Sliding window parameters
window_size = 5   # Number of messages per window
overlap = 2       # Number of overlapping messages between windows

max_seq_length = 256  # Reduce from 512 if possible


# Persistent buffer to accumulate messages across stream batches
persistent_buffer = []  

def sliding_window(chat_history):
    """
    Process chat history into sliding window chunks with overlap.
    This helps maintain conversation context over multiple messages.

    Args:
        chat_history (list): List of chat messages (JSON objects).
    
    Returns:
        list: List of overlapping chunks of messages, each formatted for tokenization.
    """
    
    global persistent_buffer  # Use a global buffer to persist across calls
    chunks = []
    
    for message in chat_history:
        # Format message as "<name>: <text>"
        speaker = message['name']
        text = message['text']
        persistent_buffer.append(f"{speaker}: {text}")
        
        # When enough messages accumulate, create a sliding window chunk
        if len(persistent_buffer) >= window_size:
            # Take the first 'window_size' messages from the buffer
            chunk = persistent_buffer[:window_size]
            chunks.append({"messages": chunk})
            
            # Keep the last 'overlap' messages to ensure continuity across windows
            persistent_buffer = persistent_buffer[-overlap:]
            
    return chunks


def process_and_tokenize_stream(dataset):
    """
    Stream the chat data from JSONL file, apply sliding window processing, and tokenize.
    Uses Hugging Face datasets streaming to prevent memory overload.
    """
    
    buffer = []  # To accumulate and batch-process sliding windows
    batch_size = 100  # Tokenize every 100 chunks to avoid memory overflow
    
    for i, data in tqdm(enumerate(dataset)):
        buffer.append(data)
        
        # Once buffer reaches batch size, apply sliding window
        if len(buffer) >= batch_size:
            all_chunks = []
            
            # Process each conversation/message batch in sliding windows
            for chat in buffer:
                chunks = sliding_window([chat])
                all_chunks.extend(chunks)
            
            # Tokenize and save in batches
            tokenize_and_save(all_chunks)
            buffer = []  # Clear buffer after processing

    # Handle remaining buffer
    if buffer:
        all_chunks = []
        for chat in buffer:
            chunks = sliding_window([chat])
            all_chunks.extend(chunks)
        tokenize_and_save(all_chunks)


def tokenize_and_save(chunks):
    """
    Tokenize overlapping chat chunks and save the results to disk in JSON format.
    Handles Hugging Face tokenizer's BatchEncoding object by converting tensors to lists.

    Args:
        chunks (list): List of chat message chunks to be tokenized.
        output_file (str): Path to save the tokenized output.
    """
    tokenized_data = []
    data = []
    
    for chunk in chunks:
        # Join chunk messages into a single block of text for tokenization
        conversation = "\n".join(chunk['messages'])
        
        # Tokenize the conversation (returns BatchEncoding object)
        tokens = tokenizer(
            conversation,
            truncation=True,
            padding="max_length",
            max_length=max_seq_length,
            return_tensors="pt"
        )
        
        
        
         # Convert BatchEncoding (tensor) to Python lists for JSON serialization
        tokens_dict = {
            key: val.cpu().tolist()[0]  # Convert tensors to lists, this is 1 item in a list to extract to avoid mismatch in fine tuning tensors
            for key, val in tokens.items()
        }
        
        tokens_dict["labels"] = tokens_dict["input_ids"].copy()
        
        tokenized_data.append(tokens_dict)
        data.append(chunk['messages'])

    # Save tokenized data to disk or use it directly
    with open(tokenized_output_file, 'a') as f:
        for entry in tokenized_data:
            json.dump(entry, f)
            f.write('\n')
            
    # Save data to disk
    with open(data_output_file, 'a') as f:
        for entry in data:
            json.dump(entry, f)
            f.write(f'\n')


# Saves to file
process_and_tokenize_stream(dataset)


0it [00:00, ?it/s]

62923it [00:37, 1684.45it/s]


In [6]:
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from datasets import Dataset
import json

# Load the tokenized data from JSONL file
def load_tokenized_data(file_path):
    data = []
    
    # Read each line (which is a tokenized entry)
    with open(file_path, 'r') as f:
        for line in f:
            # Convert JSON string back to dictionary
            data.append(json.loads(line))
    
    # Create a Hugging Face Dataset from the list of dictionaries
    return Dataset.from_list(data)

# LoRA Configuration
lora_config = LoraConfig(
    r=16,  # Low-rank dimension
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Apply LoRA to attention layers Q,V
    lora_dropout=0.1,  # Dropout for regularization
    bias="none",  # No additional bias
    task_type="CAUSAL_LM"  # Language modeling task
)


# Load tokenized dataset
tokenized_dataset = load_tokenized_data("data/tokenized_output.json")


# Prepare model for k-bit training (unfreezes LoRA adapters). This is needed if using quantization (bitsandbytes)
model = prepare_model_for_kbit_training(model)

# Wrap the base model with LoRA adapters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Confirm trainable parameters



trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [7]:
tokenizer.decode(tokenized_dataset[0]["input_ids"], skip_special_tokens=True)

'Cole Lewis: Hi!\nBrayden Turner: Ugh\nCole Lewis: Did you guys start installing\nErik Tharp: Conner isn’t responding\nErik Tharp: He said he had an interview at 4:15 so he’s probably doing that'

In [10]:
from evaluate import load

metric = load("perplexity")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    
    # Ensure logits are converted to probabilities
    if isinstance(logits, tuple):
        logits = logits[0]  # For models returning tuples

    perplexity = metric.compute(predictions=predictions, references=labels)
    perplexity["perplexity"] = np.exp(perplexity["loss"])  # Convert loss to perplexity
    return perplexity

training_args = TrainingArguments(
    output_dir="./fine-tuned-group-chat",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,
    
    save_steps=500,
    logging_steps=50,  # Log progress every 50 steps
    logging_dir="./logs",  # Path for TensorBoard logs
    logging_strategy="steps",
    report_to=["tensorboard"],  # Report metrics to TensorBoard, launch with tensorboard --logdir ./logs
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics,
)

c:\Users\Brayden Turner\Projects\chat_playground\chat_playground\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
torch.cuda.empty_cache()
model.train()
trainer.train()

In [1]:
from peft import PeftModel
from transformers import AutoTokenizer, LlamaForCausalLM, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch

torch.cuda.empty_cache()
fine_tuned_path = "fine-tuned-group-chat/experiment_lr=5e-05_2025-01-08_23-49-05/checkpoint-4981"  # Path to your fine-tuned model

bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, # 8 bit uses too much memory
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
model_id_base = "meta-llama/Meta-Llama-3.1-8B" # General model
model_id_instruct = "meta-llama/Meta-Llama-3.1-8B-Instruct" # Instruct model


# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id_base)

# Load the model with quantization and auto device mapping
model = AutoModelForCausalLM.from_pretrained(
    model_id_base,
    quantization_config=bnb_config,
    device_map="auto"  # Automatically uses GPU, with CPU fallback if necessary
)

# Attach the fine-tuned LoRA adapters
fine_tuned_model = PeftModel.from_pretrained(model, fine_tuned_path)





Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [31]:
import random

model.eval()

max_seq_length = 256
speakers = ["Conner", "Brayden", "Cole", "Erik"]

def truncate_context(context, num_exchanges=3):
    exchanges = context.strip().split("\n")
    truncated_context = "\n".join(exchanges[-num_exchanges:])
    return truncated_context


def truncate_context_with_sliding_window(context, max_length):
    """
    Truncate the context using a sliding window to fit within the maximum length.
    """
    # Tokenize the context
    tokenized_context = tokenizer.encode(context, truncation=False, return_tensors="pt")
    
    # If context exceeds max_length, truncate from the start
    if tokenized_context.size(1) > max_length:
        tokenized_context = tokenized_context[:, -max_length:]
    
    # Decode the truncated context back to text
    truncated_context = tokenizer.decode(tokenized_context[0], skip_special_tokens=True)
    return truncated_context


def generate_response(prompt, max_length=max_seq_length, temperature=0.6, top_k=50):
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate text
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=0.9, 
        do_sample=False,  # Enables sampling for more creative output
        pad_token_id=tokenizer.eos_token_id,  # Prevents errors related to pad tokens
        repetition_penalty=1.5,  # Penalize repeating tokens
        # no_repeat_ngram_size=3,         
    )
    
    # Decode the generated text
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response
    
# Inference
context = "Conner: Brayden how is your cursed simulacrum of our friendship coming along?\n"
full_response = ""

# Simulate an infinite conversation loop
for step in range(5):  # Replace 100 with `while True` for an infinite loop
    print(f"Step {step + 1}")
    print(f"Current Context:\n{context}")

    # Generate the model's response
    response = generate_response(context)
    print(f"Model Response:\n{response}\n")
    full_response += "\n".join(response.split("\n")[-2:])
    
    # Append the response to the context with speaker label
    context += f"{response.strip()}\n"
    
    # Truncate context using a sliding window
    context = truncate_context(context)
    
print(f"Full response:\n{full_response}")

Step 1
Current Context:
Conner: Brayden how is your cursed simulacrum of our friendship coming along?



c:\Users\Brayden Turner\Projects\chat_playground\chat_playground\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\Brayden Turner\Projects\chat_playground\chat_playground\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Model Response:
Conner: Brayden how is your cursed simulacrum of our friendship coming along?
Braydon Turner: I’m not sure if it’s a curse or blessing 
Cole Lewis: It could be both
Erik Tharp: ￼I don’t know what to do with my life anymore. This was the first time in awhile that we’ve been able to play together and now you guys are going back on Sunday.
None: None

Step 2
Current Context:
Cole Lewis: It could be both
Erik Tharp: ￼I don’t know what to do with my life anymore. This was the first time in awhile that we’ve been able to play together and now you guys are going back on Sunday.
None: None
Model Response:
Cole Lewis: It could be both
Erik Tharp: ￼I don’t know what to do with my life anymore. This was the first time in awhile that we’ve been able to play together and now you guys are going back on Sunday.
None: None
Conner Morton: I’m gonna go get a haircut then maybe some food 
Brayden Turner: Conners hair is so long

Step 3
Current Context:
None: None
Conner Morton: I’m gonna 